In [0]:
import os
from datetime import datetime
from pyspark.sql.functions import lit

# 1. Define the log file path within the local repository
log_path = "./silver_schema_warnings.log"

def log_warning(message):
    """Saves the log warning into a plain text file in the repository"""
    with open(log_path, "a") as log_file:
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_file.write(f"[{current_time}] WARNING: {message}\n")

print("Starting schema analysis and homologation for the Silver Layer...")

# 2. Load the raw data from Bronze
df_yellow = spark.read.parquet("/Workspace/Shared/bronze/yellow_tripdata_2026_*.parquet")
df_green = spark.read.parquet("/Workspace/Shared/bronze/green_tripdata_2026_*.parquet")

# 3. Manual homologation for dates (Known homologous names) and source tagging
df_yellow = df_yellow.withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
                     .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime") \
                     .withColumn("taxi_type", lit("yellow"))

df_green = df_green.withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
                   .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime") \
                   .withColumn("taxi_type", lit("green"))

# 4. Extract schemas into Python Sets for dynamic comparison
yellow_columns = set(df_yellow.columns)
green_columns = set(df_green.columns)

# 5. Determine the intersection (matching columns) and differences
common_columns = yellow_columns.intersection(green_columns)
yellow_exclusive = yellow_columns - common_columns
green_exclusive = green_columns - common_columns

# 6. Drop and log orphaned columns
print("\n--- Column Audit ---")
for col in yellow_exclusive:
    alert = f"Column '{col}' is discarded from the YELLOW dataset because there are no matches during the merge or its homologue is unknown."
    print(f"⚠️ {alert}")
    log_warning(alert)

for col in green_exclusive:
    alert = f"Column '{col}' is discarded from the GREEN dataset because there are no matches during the merge or its homologue is unknown."
    print(f"⚠️ {alert}")
    log_warning(alert)

# 7. Strict selection: Keep only the common core columns
df_yellow_silver = df_yellow.select([c for c in common_columns])
df_green_silver = df_green.select([c for c in common_columns])

# 8. Guaranteed error-free final unification (Merge)
df_unified_trips = df_yellow_silver.unionByName(df_green_silver)

# 9. Data Quality Rules (Silver Layer)
print("\n--- Applying Data Quality Rules ---")
df_silver_clean = df_unified_trips.filter("passenger_count > 0 AND trip_distance > 0")

# 10. Save to Silver Layer as a Databricks Managed Table
table_name = "default.silver_unified_trips"
print(f"\nWriting data to Databricks Managed Table: {table_name}...")

# Delegamos la gestión del almacenamiento al catálogo de Databricks
df_silver_clean.write.format("delta").mode("overwrite").saveAsTable(table_name)

print("✅ Silver Layer successfully consolidated as a Managed Table!")